# 10 — Complete acceptance matrix

This notebook is the final, reviewer-readable verification layer. It does **not** replace `pytest`: it connects each architectural promise to an executable scenario, its observed output, and an explicit pass criterion.

Two evidence levels are kept separate:

- **End-to-end** uses the real runtime corpus, planner, hybrid retrieval, evidence gate and response contract.
- **Controlled branch test** injects minimal synthetic candidates to prove a rare deterministic branch such as conflict or provider failure. It does not claim that this event occurred in a public Foyer document.

Online answer generation is disabled here. The cached Gemini document and query embeddings are still exercised by the real QRT scenario without spending API quota.

In [1]:
from pathlib import Path
from IPython.display import display
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table
ROOT = bootstrap()
for module_name in list(sys.modules):
    if module_name == 'app' or module_name.startswith('app.'):
        del sys.modules[module_name]

from app.domain.mapping import HybridQuestionMapper
from app.domain.models import (
    AnswerStatus, Candidate, Chunk, EvidenceConstraints, EvidenceField,
    EvidenceProfile, Fact, QuestionRequest, ScopeSelection, ScoreTrace, SourceLocator,
)
from app.evidence.gate import DeterministicEvidenceGate
from app.retrieval.dense import FallbackDenseIndex, LocalDenseIndex
from app.retrieval.references import resolve_references
from app.services.engine import EvidenceEngine, choose_retrieval_strategy, stop_reason_for_status

engine = EvidenceEngine()
engine.online_composer.api_key = ''
results = []
def record(requirement, level, observed, passed):
    results.append({'requirement': requirement, 'evidence_level': level, 'observed': observed, 'pass': bool(passed)})
    assert passed, f'Acceptance failure: {requirement} — {observed}'

## 1. Real end-to-end orchestration

The first request must retrieve three different prudential fields from the real Groupe Foyer QRT and stop immediately when the contract is complete. The second deliberately asks for the wrong period: 2025 facts must be rejected, all three retrieval depths must be attempted, and the system must abstain after exhausting its fixed budget.

In [2]:
base = {
    'question': "What public evidence describes Groupe Foyer's prudential coverage in 2025?",
    'scope': ScopeSelection(document_ids=['foyer_group_qrt_2025']),
}
complete = await engine.answer(QuestionRequest(**base, constraints=EvidenceConstraints(entity='Groupe Foyer', period='2025')))
wrong_period = await engine.answer(QuestionRequest(**base, profile_id='prudential_coverage', constraints=EvidenceConstraints(entity='Groupe Foyer', period='2024')))
runtime_rows = [
    {'case': 'real QRT / valid period', 'status': complete.status, **complete.retrieval_run.model_dump(), 'accepted': sum(e.state == 'ACCEPTED' for e in complete.evidence)},
    {'case': 'real QRT / wrong period', 'status': wrong_period.status, **wrong_period.retrieval_run.model_dump(), 'accepted': sum(e.state == 'ACCEPTED' for e in wrong_period.evidence)},
]
display(display_table(runtime_rows))
record('multi-field batch routing', 'end-to-end real QRT', complete.retrieval_run.strategy, complete.retrieval_run.strategy == 'batch_multi_field')
record('contract-complete early stop', 'end-to-end real QRT', f'{complete.status}; k={complete.retrieval_run.k_history}', complete.status == AnswerStatus.COMPLETE and complete.retrieval_run.stop_reason == 'contract_complete' and complete.retrieval_run.k_history == [1])
record('entity/period enforcement and bounded recovery', 'end-to-end real QRT', f'{wrong_period.status}; k={wrong_period.retrieval_run.k_history}', wrong_period.status == AnswerStatus.NOT_FOUND and wrong_period.retrieval_run.stop_reason == 'budget_exhausted' and wrong_period.retrieval_run.k_history == [1, 3, 5])
record('trained dense cache identified honestly', 'end-to-end real QRT', complete.retrieval_run.dense_provider, complete.retrieval_run.dense_provider == 'gemini')

,case,status,strategy,dense_provider,rounds,k_history,stop_reason,resolved_references,unresolved_references,accepted
0,real QRT / valid period,COMPLETE,batch_multi_field,gemini,1,[1],contract_complete,[],[],3
1,real QRT / wrong period,NOT_FOUND,batch_multi_field,gemini,3,"[1, 3, 5]",budget_exhausted,[],[],0


## 2. Rare deterministic branches

A one-field contract demonstrates `sequential_top1`. Two incompatible values for the same field, entity and period demonstrate `CONFLICT`. These are controlled fixtures because manufacturing a conflict inside a public source would be misleading. The evidence gate—not an LLM—makes both decisions.

In [3]:
field = EvidenceField(id='ratio', label='SCR coverage ratio', value_type='percentage', query_templates=['{question}'])
profile = EvidenceProfile(id='single_ratio', version='1', label='Single ratio', description='Controlled one-field contract', triggers=[], fields=[field])
def candidate(chunk_id, value):
    locator = SourceLocator(document_id='controlled', document_title='Controlled fixture', version='1', source_url='synthetic://acceptance', page=1, section_path=['1.1'])
    fact = Fact(field_id='ratio', label=field.label, value=value, formatted_value=f'{value}%', value_type='percentage', unit='%', period='2025', entity='Groupe Foyer')
    chunk = Chunk(id=chunk_id, document_id='controlled', text=f'SCR coverage ratio: {value}%.', locator=locator, facts=[fact])
    return Candidate(field_id='ratio', chunk=chunk, score=ScoreTrace(rrf_score=1.0))

gate = DeterministicEvidenceGate()
single = gate.evaluate(profile, [candidate('ratio-a', 201.0)], EvidenceConstraints(entity='Groupe Foyer', period='2025'))
conflict = gate.evaluate(profile, [candidate('ratio-a', 201.0), candidate('ratio-b', 202.0)], EvidenceConstraints(entity='Groupe Foyer', period='2025'))
display(display_table([
    {'case': 'one accepted fact', 'fields': len(profile.fields), 'strategy': choose_retrieval_strategy(len(profile.fields)), 'status': single.status, 'stop_reason': stop_reason_for_status(single.status)},
    {'case': 'two incompatible facts', 'fields': len(profile.fields), 'strategy': choose_retrieval_strategy(len(profile.fields)), 'status': conflict.status, 'stop_reason': stop_reason_for_status(conflict.status)},
]))
record('single-field sequential routing', 'controlled branch test', choose_retrieval_strategy(1), choose_retrieval_strategy(1) == 'sequential_top1' and single.status == AnswerStatus.COMPLETE)
record('conflict terminal stop', 'controlled branch test', f'{conflict.status}; {stop_reason_for_status(conflict.status)}', conflict.status == AnswerStatus.CONFLICT and stop_reason_for_status(conflict.status) == 'conflict')

,case,fields,strategy,status,stop_reason
0,one accepted fact,1,sequential_top1,COMPLETE,contract_complete
1,two incompatible facts,1,sequential_top1,CONFLICT,conflict


## 3. Controlled explicit-reference resolution

Only explicit `see section …` / `voir section …` references are followed, only inside the same document, and at most three per candidate batch. Resolved and unresolved targets are returned as trace data. Semantic or cross-document references are deliberately out of scope.

In [4]:
def plain_chunk(chunk_id, text, section):
    locator = SourceLocator(document_id='reference-doc', document_title='Reference fixture', version='1', source_url='synthetic://references', page=1, section_path=[section])
    return Chunk(id=chunk_id, document_id='reference-doc', text=text, locator=locator)
source = plain_chunk('source', 'See section 1.1, voir section 1.2, see section 1.3 and see section 1.4.', '1.0')
targets = [plain_chunk(f'target-{index}', f'Target {index}', f'1.{index}') for index in range(1, 5)]
resolution = resolve_references([Candidate(field_id='ratio', chunk=source, score=ScoreTrace(rrf_score=1.0))], [source, *targets])
unresolved_source = plain_chunk('missing-source', 'See section 9.9.', '2.0')
unresolved = resolve_references([Candidate(field_id='ratio', chunk=unresolved_source, score=ScoreTrace())], [unresolved_source])
display({'resolved': resolution.resolved, 'fourth_target_followed': any('target-4' in item for item in resolution.resolved), 'unresolved': unresolved.unresolved})
record('maximum three references per batch', 'controlled branch test', resolution.resolved, len(resolution.resolved) == 3 and not any('target-4' in item for item in resolution.resolved))
record('resolved and unresolved references exposed', 'controlled branch test', unresolved.unresolved, len(resolution.resolved) == 3 and len(unresolved.unresolved) == 1)

{'resolved': ['source -> section 1.1 -> target-1',
  'source -> section 1.2 -> target-2',
  'source -> section 1.3 -> target-3'],
 'fourth_target_followed': False,
 'unresolved': ['missing-source -> section 9.9']}

## 4. Ambiguity abstention and dense-provider failure

Entity and period recognition is explicit alias/year extraction. Multiple detected values cause abstention unless the caller disambiguates. Separately, a deliberately failing Gemini-like primary proves that retrieval remains available through the clearly labelled hashing baseline.

In [5]:
mapper = HybridQuestionMapper()
mapper.online_enabled = False
multi_entity = mapper.map('Compare Groupe Foyer and Foyer Assurances in 2025')
multi_period = mapper.map('Compare Groupe Foyer prudential coverage in 2024 and 2025')
class FailingPrimary:
    provider = 'gemini'
    def search(self, query):
        raise RuntimeError('controlled provider outage')
fallback_chunk = plain_chunk('fallback', 'Groupe Foyer SCR coverage ratio', '1.0')
dense = FallbackDenseIndex(FailingPrimary(), LocalDenseIndex([fallback_chunk]))
dense_results = dense.search('SCR coverage')
display(display_table([
    {'case': 'multiple entities', 'profile': multi_entity.profile.id, 'decision': multi_entity.decision.decision_source, 'ambiguities': multi_entity.decision.ambiguities},
    {'case': 'multiple periods', 'profile': multi_period.profile.id, 'decision': multi_period.decision.decision_source, 'ambiguities': multi_period.decision.ambiguities},
    {'case': 'primary dense outage', 'profile': '-', 'decision': dense.provider, 'ambiguities': []},
]))
record('multi-entity abstention', 'mapping integration test', multi_entity.decision.ambiguities, multi_entity.profile.id == 'open_question' and multi_entity.decision.decision_source == 'abstention')
record('multi-period abstention', 'mapping integration test', multi_period.decision.ambiguities, multi_period.profile.id == 'open_question' and multi_period.decision.decision_source == 'abstention')
record('dense failure fallback is explicit', 'controlled branch test', dense.provider, bool(dense_results) and dense.provider == 'gemini+hashing-fallback')

,case,profile,decision,ambiguities
0,multiple entities,open_question,abstention,"[multiple entities detected: Groupe Foyer, Foy..."
1,multiple periods,open_question,abstention,"[multiple periods detected: 2024, 2025]"
2,primary dense outage,-,gemini+hashing-fallback,[]


## 5. Final acceptance decision

Every row below is asserted in the cell that produced it. A green matrix means the specified demo behaviours work on this version; it does not prove universal PDF extraction, semantic reference resolution, unrestricted scale, or model quality outside the reviewed corpus.

In [6]:
display(display_table(results))
summary = {'checks': len(results), 'passed': sum(row['pass'] for row in results), 'failed': sum(not row['pass'] for row in results)}
display(summary)
assert summary['checks'] == 11
assert summary['failed'] == 0
print('ACCEPTANCE RESULT: PASS — all explicitly scoped demo requirements above are verified.')

,requirement,evidence_level,observed,pass
0,multi-field batch routing,end-to-end real QRT,batch_multi_field,True
1,contract-complete early stop,end-to-end real QRT,COMPLETE; k=[1],True
2,entity/period enforcement and bounded recovery,end-to-end real QRT,"NOT_FOUND; k=[1, 3, 5]",True
3,trained dense cache identified honestly,end-to-end real QRT,gemini,True
4,single-field sequential routing,controlled branch test,sequential_top1,True
5,conflict terminal stop,controlled branch test,CONFLICT; conflict,True
6,maximum three references per batch,controlled branch test,"[source -> section 1.1 -> target-1, source -> ...",True
7,resolved and unresolved references exposed,controlled branch test,[missing-source -> section 9.9],True
8,multi-entity abstention,mapping integration test,"[multiple entities detected: Groupe Foyer, Foy...",True
9,multi-period abstention,mapping integration test,"[multiple periods detected: 2024, 2025]",True


{'checks': 11, 'passed': 11, 'failed': 0}

ACCEPTANCE RESULT: PASS — all explicitly scoped demo requirements above are verified.
